# 第1回 演習：データを知り、明日の台数に答える（解答例・教員用）


---


## 今日の分析目標


あなたは自転車シェアの運営者です。明日、自転車を何台用意すればいいかに、数字で答える。

まず表を読み、1日の台数がどれくらい振れるか、どんな手がかりと関係していそうかを確かめます。そのうえで、条件が似た日の平均で明日の台数に答え、どれだけ外れるかで当たり具合を確かめます。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは自転車シェアの運営者です。街のあちこちに自転車を置いて借りてもらい、好きな場所で乗り捨ててもらうサービスです。

決めたいのは、明日に向けて自転車を何台用意するかです。手がかりは、明日の季節・天気・気温などです。自転車が足りなければ、借りたい人を逃します。余れば、置き場所と整備の手間がむだになります。足りなくても余っても困るので、毎晩この台数を決めています。


### 手元のデータ

ある自転車シェアの、2011〜2012年の記録です（UCI Machine Learning RepositoryのBike Sharing Dataset）。答えの列は、その日の利用台数`cnt`です。1行が何を表すかと列の中身は、1節で表を読み込んでから確かめます。


### 最後に出す答え

明日と条件が似た日の平均から出す「およそ何台」という1つの数と、その数が平均して何台くらいずれるかです。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- Colabでノートを開き、セルを上から実行し、エラーのいちばん下の行を読める
- pandasで表を読み込み、列を選び、条件で行を絞り、平均などを集計できる
- `describe()`・ヒストグラム・散布図から、目的変数（response variable）の振れ方と、手がかりの形・癖を言える
- 相関（correlation）で手がかりの順位を下見し、相関が因果ではないことを説明できる
- 条件が似た日の平均（`groupby`）で台数を当て、「外れ幅の平均」で当たり具合を測れる。条件を細かくしすぎたときの落とし穴を説明できる


---


## 0. Colabを動かす


なぜColabを使うのか　Pythonの道具（パッケージ）は互いのバージョンに依存していて、パッケージAを新しくしたらBが対応しておらずインストールに失敗する、ということがよく起こります。そのため実務や研究では、解析プロジェクトごとに環境を分けて作るのが一般的です（DockerやApptainerのようなコンテナ、Anacondaの仮想環境など）。この授業では、その準備を省くためにColabを使います。主要なパッケージが最初からそろった環境をGoogleが用意しているので、ブラウザを開くだけで全員が同じ環境になります。また、AzureやAWSのようなクラウドと違って無料の範囲で使え、上限に達しても一時的に使えなくなるだけで、勝手に課金されることはありません。

ノートを開いてコピーする　教材ページの表で、その回の「演習」列の［開く］をクリックすると、Colabでノートが開きます。開いた直後のノートは借り物で、書き込んでも保存されません。［ファイル］→［ドライブにコピーを保存］（上の帯の［ドライブにコピー］でも同じ）で、自分のノートにしてから始めます。

セルを実行する　ノートは、文章のセルとコードのセルの並びです。コードのセルは左の ▶ かShift + Enterで実行し、結果はすぐ下に出ます。終わるとセルの左に ✓ が付きます。最初の1回は、Colabの準備と道具の取り寄せで十数秒かかります。

プログラムは上から下へ　1つのセルの中は上の行から順に、ノートは上のセルから順に実行します。前のセルで作ったもの（読み込んだ表など）を、後のセルで使うからです。途中のセルを飛ばすと、後のセルは「そんな名前は知らない」と止まります。

エラーは、いちばん下の行を読む　赤い表示が出たら、いちばん下の行に原因が書いてあります。よく出るのは次の3つです。

- `NameError`：名前が見つからない。たいていは前のセルを飛ばした
- `ModuleNotFoundError`：使おうとした道具（ライブラリ）が入っていない
- `SyntaxError`：書き方の誤り。打ち間違い

それでも動かないとき　［ランタイム］→［ランタイムを再起動］で、それまでに実行したことをいったん忘れさせ、［ランタイム］→［すべてのセルを実行］で上から順にやり直します。


### お手本のセルの読み方

このノートでは、各節でまずお手本のセルを実行して読み方を確かめ、そのあとでそのセルを少し変えて広げていきます。セルは1行ずつ追うより、先に「何のためのかたまりか（役割）・何を受け取り（入力）・何を出すか（出力）」をつかむと読みやすくなります。

TODOを埋める　TODOのセルにある`...`が、あなたが書く場所です。何を書くかはセルの前の文章に、手がかりはセルの中の「ヒント」のコメントにあります。たいていは、すぐ上のお手本のセルを少し変えれば書けます。


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：「セットアップ完了」の表示

表を読む機能や図を描く機能はPythonの本体には入っていないので、ライブラリを足して使います。`import pandas as pd`は、表を扱うpandasを`pd`という略称で使えるようにする書き方です。`!pip install`は、Colabに入っていない道具（ここでは図の日本語表示）を取り寄せる命令です。何よりも先に、このセルを実行してください。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、この3つを一周します。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for x, p, c in zip([0.17, 0.5, 0.83], phases, colors):
    ax.text(x, 0.62, p, ha='center', va='center', fontsize=13, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95))
for x0, x1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(x1, 0.62), xytext=(x0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. どんなデータが手元にある？


明日の台数に答えるために、まず手元のデータを見ます。手元にあるのは、ある自転車シェアの2年ぶん（2011〜2012年）の記録です。まず表を読み込んで、どんな形をしているかを見ます。


### お手本：表を読み込んで、先頭を見る

- 役割：CSVファイルの表を読み込み、使わない列を落として、先頭の5行を見る
- 入力：データの置き場所`DATA_DIR`と、ファイル名`bike_day.csv`
- 出力：表`df`（このあと全部の節で使う）と、その先頭5行

`df = pd.read_csv(...)`は、右で読み込んだ表を、左の名前`df`にしまっておく書き方です。こうして名前を付けてしまっておくものを変数と呼びます。`pd.read_csv`は「pandasのread_csv」という関数です。`df.head()`のように`df.`に続けて書くのは、表そのものが持っている機能（メソッド）の呼び出しです。


In [ ]:
# 自転車シェアの日次データ（1日1行）を読み込む
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
df = pd.read_csv(f'{DATA_DIR}/bike_day.csv')

# instant(通し番号)・dteday(日付)・casual/registered は今回は使わないので落とす
#   ※ casual + registered = cnt（利用台数）なので、残すと答えが最初から分かってしまう
df = df.drop(columns=['instant', 'dteday', 'casual', 'registered'])
df.head()


1行が1日です。当てたい値は`cnt`（その日の利用台数）で、残りの列が手がかりの候補になります。`casual`（一般客）と`registered`（会員）は、足すとちょうど`cnt`になる内訳なので落としました。残すと「台数を当てる」問いの答えが、最初から分かってしまうからです。

| 列 | 意味 |
|---|---|
| `season` | 季節の区分（1〜4） |
| `yr` | 年（0＝2011年、1＝2012年） |
| `mnth` | 月（1〜12） |
| `holiday`・`workingday` | 祝日か（1/0）、平日か（1/0） |
| `weekday` | 曜日（0〜6） |
| `weathersit` | 天気（1＝晴れ〜3＝雨・雪） |
| `temp`・`atemp` | 気温・体感気温（0〜1に縮めた値） |
| `hum`・`windspeed` | 湿度・風速（0〜1に縮めた値） |
| `cnt` | その日の利用台数（当てたい値） |


### 変えてみる：表の大きさを見る

`head()`は先頭だけでした。表全体の大きさは`shape`で分かります。`shape`はメソッドではなく、表が持っている値（属性）なので、`()`を付けません。


In [ ]:
print(df.shape)       # (行の数, 列の数)
print(df.shape[0])    # 行の数だけ。[0] は「1つ目」


731行×12列。2年ぶんの日数が、1日も欠けずに並んでいます。


---


## 2. 1日の台数はどれくらい振れる？


1行が1日で、731日ぶんの記録がそろいました。明日の台数を当てる前に、そもそも1日の台数がどれくらい振れるのかを知っておきます。振れが小さければ、毎日同じ台数を用意すれば済むからです。


### お手本：1つの列を選んで、要約する

- 役割：台数の列だけを取り出し、平均やばらつきにまとめる
- 入力：表`df`
- 出力：`cnt`の件数・平均・標準偏差・最小・分位点（quantile）・最大

`df['cnt']`のように`[ ]`に列名を入れると、その列だけを取り出せます。`describe()`は、列を数行の要約にまとめるメソッドです。


In [ ]:
df['cnt'].describe().round(1)


読み方　countは件数、meanは平均、stdは標準偏差（standard deviation。典型的なズレ幅で、大きいほど日ごとの差が激しい）です。min / 25% / 50% / 75% / maxは、値を小さい順に並べたときの位置（分位点）で、真ん中の50%が中央値です。

平均は4504台、標準偏差は1937台。日によって、最小22台から最大8714台まで大きく振れます。毎日同じ台数を用意するのでは、足りない日も余る日も多そうです。


### 変えてみる：全部の列を要約する

`df['cnt']`の`['cnt']`を外して`df.describe()`にすると、全部の列を一度に要約できます。


In [ ]:
df.describe().round(2)     # ['cnt'] を外すと、全部の列


### 変えてみる：欠けている値を数える

実データでは、記録が抜けている（欠損値、missing value）ことがよくあります。`isnull()`は各マスが欠けているかをTrue/Falseで表し、`sum()`で列ごとにTrueの数を数えます。


In [ ]:
df.isnull().sum()          # 列ごとに、欠けた値の数


すべての列で0。このデータには欠損がありません。実務のデータでは、欠損は普通にあります。


### TODO①：気になる列を挙げてみよう

上の`df.describe()`の出力を見て、「あれ？」と思う列や、性質の似ている列を1つ以上挙げ、理由をコメントに書いてください。コードで何かを計算する必要はありません。


In [ ]:
# TODO: describe() の出力で、気になる列とその理由をコメントに書く

# 解答例①：気になる列の例
# - temp（気温）と atemp（体感気温）が非常によく似ている → ほぼ同じ情報を持つ2列
# - season(1-4) は数値だが大小に意味のない区分の番号
# - weathersit(1-3) は順だけを表す番号 → そのまま計算してよいか注意
print('気になる列の例: temp と atemp がそっくり / season・weathersit は数値の顔をした区分')


2つの癖　1つ目は、`temp`と`atemp`の平均（0.50と0.47）もばらつき（0.18と0.16）もほとんど同じことです。中身がほぼ重なった2列です。

2つ目は、`season`（1〜4）や`weathersit`（1〜3）が数字で入っていても、量を表していないことです。`season=4`は`season=1`の4倍ではなく、区分に付けた番号にすぎません。`weathersit`は1晴れから3雨・雪への順を表しますが、その間隔に意味はありません。今日は、数字のまま進めます。


---


## 3. 台数と手がかりは、どんな形？


数字の要約だけでは、値がどう散らばっているか（形）までは見えません。図で確かめます。


### お手本：台数のヒストグラム

- 役割：台数を区間に分け、各区間に何日あったかを棒で描く
- 入力：台数の列`df['cnt']`
- 出力：ヒストグラム（横軸が台数、縦軸が日数）

`plt.hist`の`bins=40`は、区間の数です。


In [ ]:
plt.hist(df['cnt'], bins=40, color='#0066cc', edgecolor='white')
plt.xlabel('1日の利用台数 cnt')
plt.ylabel('日数')
plt.show()


少ない日から多い日まで、なだらかに広がっています。平均（4504）と中央値（4548）がほぼ同じで、極端に片寄った形でもありません。


### 変えてみる：手がかりの列も描く

`df['cnt']`を`df['temp']`に変えれば、気温のヒストグラムになります。下のセルは、それを8本の列について繰り返しています。


In [ ]:
features = ['temp', 'atemp', 'hum', 'windspeed',
            'season', 'weathersit', 'mnth', 'yr']
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.flat, features):
    ax.hist(df[col], bins=30, color='#2a9d8f', edgecolor='white')
    ax.set_title(col, fontsize=11)
plt.tight_layout()
plt.show()


形が2種類あります。`temp`・`atemp`・`hum`・`windspeed`はなめらかに散らばる連続値、`season`・`weathersit`・`yr`などは数本の棒しか立たない、とびとびの値です。


### 変えてみる：気温と台数の散布図

1つの列の形から、2つの列の関係の形に進みます。`plt.scatter(横, 縦)`は、1日を1つの点にして描きます。`temp`は0〜1に縮めた値なので、41を掛けて℃に戻しています。


In [ ]:
plt.scatter(df['temp'] * 41, df['cnt'], s=8, alpha=0.3)
plt.xlabel('気温（℃）')
plt.ylabel('利用台数 cnt')
plt.show()


全体として右上がりで、暖かい日ほど台数が多い傾向があります。ただ、いちばん暑いあたりでは伸びが止まり、少し下がっているようにも見えます。暑すぎる日は、かえって自転車が敬遠されるのかもしれません。関係は右上がりでも、一直線とは限らないのです。


### TODO②：好きな列で散布図を描く

`features`の中から気になる列を1つ選び、`cnt`との散布図を描いてください。点の散らばり方から、その列と台数の関係について気づいたことを考えてみましょう。


In [ ]:
# TODO: features から好きな列を1つ選び、cnt との散布図を描く

# 解答例②：湿度 hum と利用台数の散布図
plt.scatter(df['hum'], df['cnt'], s=8, alpha=0.3)
plt.xlabel('湿度 hum（0〜1）')
plt.ylabel('利用台数 cnt')
plt.show()


---


## 4. 暑い日と寒い日で、台数は違う？


散布図では、暖かい日ほど台数が多そうでした。これを数字で確かめます。暑い日だけを取り出して、その日の台数の平均を出せばよいはずです。


### お手本：条件で行を絞る

- 役割：25℃以上の日だけを取り出し、その日数と台数の平均を出す
- 入力：表`df`
- 出力：暑い日だけの表`hot`、その日数と平均台数

`df['temp'] * 41 >= 25`は、1日ごとに「25℃以上か」をTrue/Falseで並べたものです。それを`df[ ]`に入れると、Trueの行だけが残ります。たとえば112〜116行目の日は気温が18.9・23.8・24.9・25.9・25.4℃で、25℃以上の115・116行目だけが`hot`に残ります。


In [ ]:
hot = df[df['temp'] * 41 >= 25]     # 25度以上の日だけ
print('日数:', len(hot))
print('平均台数:', round(hot['cnt'].mean()))


25℃以上の日は242日、平均は5632台。全体の平均4504台より、1000台以上多い。


### 変えてみる：寒い日と比べる

条件の`>= 25`を`< 10`に変えれば、10℃未満の日になります。


In [ ]:
cold = df[df['temp'] * 41 < 10]     # 10度未満の日だけ
print('日数:', len(cold))
print('平均台数:', round(cold['cnt'].mean()))


10℃未満の日は60日、平均は1678台。暑い日の3分の1もありません。散布図で見えた「暖かいほど多い」が、数字でも確かめられました。


### 変えてみる：条件を2つ重ねる

条件は重ねられます。`&`は「かつ」で、それぞれの条件を`( )`で囲みます。暑い日のうち、2年目（`yr`が1）の日だけを取り出してみます。


In [ ]:
# 25度以上 かつ 2年目
hot2 = df[(df['temp'] * 41 >= 25) & (df['yr'] == 1)]
print('日数:', len(hot2))
print('平均台数:', round(hot2['cnt'].mean()))


暑い2年目の日は120日、平均6741台。同じ暑い日でも、2年目のほうがさらに多い。気温だけでなく、年も関係していそうです。`==`は「等しいか」を調べる記号で、`=`（名前にしまう）とは別物です。


### TODO③：天気で比べる

晴れの日（`weathersit`が1）と、雨・雪の日（`weathersit`が3）で、日数と平均台数を比べてください。


In [ ]:
# TODO: 晴れの日（weathersit == 1）と雨・雪の日（== 3）で、

# 解答例③：晴れの日と雨・雪の日
sunny = df[df['weathersit'] == 1]
rainy = df[df['weathersit'] == 3]
print('晴れ   日数:', len(sunny), ' 平均台数:', round(sunny['cnt'].mean()))
print('雨・雪 日数:', len(rainy), ' 平均台数:', round(rainy['cnt'].mean()))


---


## 5. どれがいちばん効きそう？


気温・年・天気と、台数に関係していそうな手がかりが見えてきました。では、手がかりの中でどれがいちばん台数と一緒に動くのか。相関係数（correlation coefficient）で順位を付けて、下見をします。


### お手本：台数との相関を、強い順に並べる

- 役割：各列と台数の相関係数を計算し、大きい順に並べる
- 入力：表`df`
- 出力：列ごとの相関係数（−1〜1）の並び

`df.corr(numeric_only=True)`は、数値の列どうしのすべての組み合わせについて相関係数を表にします。その`['cnt']`の列を取り出し、`drop('cnt')`で自分自身（相関1）を外し、`sort_values(ascending=False)`で大きい順に並べます。


In [ ]:
corr = df.corr(numeric_only=True)['cnt'].drop('cnt')
corr.sort_values(ascending=False).round(2)


上位は`atemp` 0.63・`temp` 0.63・`yr` 0.57・`season` 0.41。逆向き（負）には`weathersit` −0.30・`windspeed` −0.23。4節で確かめた気温・年・天気が、そのまま上位に並びました。そっくりな2列`atemp`と`temp`が同点なのも、TODO①の通りです。


### 変えてみる：そっくりな2列の相関

TODO①で「平均もばらつきもほぼ同じ」と見た`temp`と`atemp`は、どれくらい一緒に動くのでしょうか。`df['temp'].corr(df['atemp'])`のように、列`.corr(`別の列`)`と書くと、2列だけの相関係数が出ます。


In [ ]:
round(df['temp'].corr(df['atemp']), 3)


0.992。ほぼ1で、2列はほとんど同じ動きをします。こうした2列を両方とも手がかりに使うと、どちらの列の影響か決めきれません。


### 相関は、因果ではない

「気温と台数の相関が0.63」から、「気温が高いことが原因で台数が増える」とまでは言えません。季節（`season`）は、気温と台数の両方を同時に動かします。暖かい季節は気温が高く、同じ季節には出かける人も多い。すると「気温が高い日に台数が多い」の一部は、季節という共通の原因が作ったものかもしれません。こうした背後の共通の原因を交絡（confounding）と呼びます。

だから相関は、どの手がかりを疑うべきかの下見であって、因果の証明ではありません。


---


## 6. 似た日の平均で当てる


5節までの下見で、気温・年・天気が手がかりになりそうだと分かりました。いよいよ「明日、何台？」に答えます。明日の台数を出す式は、まだありません。でも、明日と条件が似た日は、過去に何台だったかなら計算できます。4節で、25℃以上の日の平均を出したのと同じことです。


### お手本：まず、何も考えずに答えると

- 役割：毎日「全体の平均」と答えたら、どれだけ外れるかを測る
- 入力：台数の列`df['cnt']`
- 出力：答え（全体の平均）と、外れ幅の平均

実際の台数と答えの差の大きさ（多すぎても少なすぎても、大きさだけを見る）を全日で平均したものを、外れ幅の平均と呼ぶことにします。`abs()`は、差の符号を外して大きさにするメソッドです。


In [ ]:
pred = df['cnt'].mean()     # 答え：毎日、全体の平均
print('答え:', round(pred), '台')
print('外れ幅の平均:', round((df['cnt'] - pred).abs().mean()), '台')


毎日4504台と答えると、外れ幅の平均は1582台になります。これを基準にして、手がかりでどこまで縮められるかを見ていきます。


### 変えてみる：気温帯ごとの平均を、まとめて出す

4節では、25℃以上の日を1回ずつ絞りました。これを全部の気温帯について一度にやるのが`groupby`です。まず`pd.cut`で気温を気温帯に分け（おおむね5℃刻み。両端の0〜10℃と30〜40℃だけは10℃幅）、`groupby('気温帯')`で同じ気温帯の日をまとめて平均します。


In [ ]:
bins = [0, 10, 15, 20, 25, 30, 40]     # おおむね5度刻み
df['気温帯'] = pd.cut(df['temp'] * 41, bins=bins)
g = df.groupby('気温帯', observed=True)['cnt']
g.agg(['mean', 'count']).round(0)


10℃以下の1678台（4節の寒い日と同じ）から、25〜30℃の5726台まで上がり、30℃を超えると5410台に下がります。散布図で見た「暑すぎると頭打ち」が、ここにも出ています。


### 変えてみる：気温帯の平均を、答えにする

この平均を、1日1日の答えに配ります。`transform('mean')`は、グループの平均を、そのグループの1日1日に配り直すメソッドです。27℃の日には、25〜30℃の平均が答えとして付きます。


In [ ]:
pred = df.groupby('気温帯', observed=True)['cnt'].transform('mean')
print('外れ幅の平均:', round((df['cnt'] - pred).abs().mean()), '台')


外れ幅は1582台から1211台に縮みました。


### 変えてみる：年と天気でも分ける

4節で、同じ暑い日でも2年目のほうが多いことを見ました。`groupby`に列のリストを渡すと、条件の組み合わせでグループができます。


In [ ]:
for keys in [['気温帯', 'yr'], ['気温帯', 'yr', 'weathersit']]:
    pred = df.groupby(keys, observed=True)['cnt'].transform('mean')
    print(keys, '→', round((df['cnt'] - pred).abs().mean()), '台')


気温帯×年で762台、基準の1582台からほぼ半分になりました。天気も足すと665台。縮みはしますが、縮み方は小さくなってきます。


### TODO④：外れ幅を縮めよう

分ける列を自分で選んで、外れ幅の平均をどこまで縮められるか試してください。使える列は`気温帯`・`season`・`yr`・`mnth`・`holiday`・`weekday`・`workingday`・`weathersit`です。いくつか組み合わせを試し、いちばん小さくなった組み合わせと、そのときの外れ幅を記録しましょう。


In [ ]:
# TODO: 分ける列のリストを keys に入れ、外れ幅の平均を計算する

# 解答例④：気温帯 × 年 × 月
keys = ['気温帯', 'yr', 'mnth']
pred = df.groupby(keys, observed=True)['cnt'].transform('mean')
print(keys, '→', round((df['cnt'] - pred).abs().mean()), '台')


### 細かく分けるほど、当たる？

条件を足していくと、外れ幅はどんどん縮みます。たとえば、月×年×天気×曜日で分けてみましょう。


In [ ]:
keys = ['mnth', 'yr', 'weathersit', 'weekday']
g = df.groupby(keys, observed=True)['cnt']
pred = g.transform('mean')
print('外れ幅の平均:', round((df['cnt'] - pred).abs().mean()), '台')
print('グループの数:', g.ngroups)
print('1日しかないグループ:', int((g.size() == 1).sum()))


外れ幅は408台まで縮みました。でも、326グループのうち101グループは1日しかありません。1日だけのグループの「平均」は、その日の台数そのものです。つまりその日は、自分自身の台数を答えにして採点していることになり、外れが0になるのは当然です。当たったとは言えません。

今日の採点は、すべて平均を作るのに使った日そのもので行っています。まだ来ていない明日で試したら、細かく分けた答えほど外れるかもしれません。


---


## 7. 運営者への答え


細かくしすぎない気温帯×年×天気で、答えを出します。明日が25〜30℃・2年目・晴れ（`weathersit`が1）なら、過去の同じ条件の日は何台だったでしょうか。4節と同じく、条件を3つ重ねて絞ります。

`pd.cut`が作った気温帯は、`(25, 30]`（25℃より上、30℃以下）のような区間です。`pd.Interval(25, 30)`と書くと、この区間を表せるので、`df['気温帯'] == pd.Interval(25, 30)`で「25〜30℃の気温帯の日か」を調べられます。最後の行は、同じ条件の日の中で、その平均を答えにしたときの外れ幅の平均です。


In [ ]:
same = df[(df['気温帯'] == pd.Interval(25, 30))
          & (df['yr'] == 1) & (df['weathersit'] == 1)]
print('同じ条件の日:', len(same), '日')
print('平均:', round(same['cnt'].mean()), '台')
print('その中での外れ幅の平均:',
      round((same['cnt'] - same['cnt'].mean()).abs().mean()), '台')


同じ条件の日は54日、平均7198台。その54日の中でも、平均で493台は外れます。答えは「およそ7200台。平均して500台くらいずれる」。明日何台かに、数字で答えられました。

ただし、まだ言えないことも残っています。

- 本当の実力は分からない：作るのに使った日で採点している。まだ来ていない日で試すとどうなるか
- 答えがガタガタの段：気温帯の境目で答えが跳ぶ。24.9℃と25.1℃で答えが変わってしまう。気温を入れると台数が出てくる、1本の式が欲しい
- グループが足りない：条件を足すほど、1つのグループの日数が減り、平均が頼りなくなる
- 因果は言えない：「暖かくすれば客が増える」とは言えない。季節が気温と台数の両方を動かしているかもしれない


---


## 目標に答えられたか


- 今日の目標は「明日、自転車を何台用意すればいいかに、数字で答える」ことでした。7節の答えは何台でしたか？ その答えは、平均してどれくらいずれるでしょうか？
- 5節の相関で、台数と最も関係が強そうな列は何だったでしょうか？6節で最初に気温で分けたのは、なぜでしたか？
- TODO④で、外れ幅の平均はどこまで縮みましたか？ その組み合わせの答えは、まだ来ていない明日でも同じくらい当たると言えるでしょうか？
- TODO①で気づいた「そっくりな2つの列」や「数値の顔をした区分」は、今後の分析でどう扱うべきだと思いますか？


---


## 今日の要点


- 大きな問い（明日何台？）は、データで答えられる形（似た日は過去に何台だった？）に言い換えると計算できる
- 当てる前に、当てたい値（`cnt`）の振れ方と、手がかりの形・癖を見ておく。散布図は、相関係数だけでは見落とす頭打ち（暑すぎる日）も教えてくれる
- 相関は「何を疑うべきか」の下見であって、因果の証明ではない
- 当てる前に基準（毎日平均と答えると1582台外れる）を置くと、手がかりの価値を外れ幅で測れる
- 手がかりで分けた平均を答えにすると外れ幅は縮む。気温帯×年の平均で762台と、ほぼ半分になった
- 条件を細かくしすぎると1日だけのグループができ、自分自身を答えにして採点してしまう
- pandasの基本は4つ：表を読む（`read_csv`）、列を選ぶ（`df['cnt']`）、条件で絞る（`df[条件]`）、まとめて集計する（`groupby`）


---


## 次回へ


今日は、気温帯・年・天気で分けた平均で「明日何台か」に答え、外れ幅の平均を基準の1582台から半分近くまで縮めました。ただ、その答えは気温帯の境目で跳ぶガタガタの段で、しかも作るのに使った日で採点していました。

次回は、この段を1本の式にします。気温を入れると台数が出てくる直線の式を立て、「良い直線」を選ぶ物差しを考えます。その式は、まだ来ていない日にも当たるのでしょうか。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

1節で読み込んだ`bike_day.csv`は1日1行のデータでした。同じ自転車シェアには、1時間1行のデータ`bike_hour.csv`もあります（17,379行。列はbike_dayと同じで、時刻を表す`hr`（0〜23）が加わります）。

`pd.read_csv(f'{DATA_DIR}/bike_hour.csv')`で時間別データを読み込み（`df`は後で使うので上書きせず、`hour`など別の名前にしてください）、時刻`hr`ごとの利用台数`cnt`の平均を求めて、横軸を時刻・縦軸を平均台数にした折れ線グラフを描いてください。集計には`groupby`（6節で使ったもの）を使います。使い方は「詰まったら」にあります。


In [ ]:
hour = pd.read_csv(f'{DATA_DIR}/bike_hour.csv')
print(f'行数: {hour.shape[0]}, 列数: {hour.shape[1]}')

by_hr = hour.groupby('hr')['cnt'].mean()
by_hr.plot(marker='o', color='#0066cc')
plt.xlabel('時刻 hr')
plt.ylabel('平均利用台数')
plt.title('時刻ごとの平均利用台数')
plt.xticks(range(0, 24, 2))
plt.grid(alpha=0.3)
plt.show()
print(by_hr.round(1))


<details><summary>詰まったら</summary>

「時刻ごとの平均」は`hour.groupby('hr')['cnt'].mean()`で求まります（時刻を添字にしたSeriesになります）。
そのSeriesに`.plot(marker='o')`をつけると折れ線グラフになります。

</details>


### 応用②（判断）

応用①の結果で、平均利用台数が最も多いのは何時ですか。時刻は、0〜23の整数で答えてください。


In [ ]:
print('最大の平均台数:', round(by_hr.max(), 1))
print('答え:', int(by_hr.idxmax()))


<details><summary>詰まったら</summary>

Seriesの`.idxmax()`は、最大値の位置（ここでは時刻）を返します。値そのものは`.max()`です。

</details>


### 応用③（解釈）

応用①のグラフを根拠に、日次データ（bike_day）だけを見ていたときには見えなかった、利用の時間帯パターンとそこから考えられることを、自転車シェアの運営担当者に向けて3行程度で書いてください。


（模範例）

利用は1日の中で一様ではなく、朝8時（平均359台）と夕方17〜18時（約462台・約426台）に2つの山があり、深夜4時は6台まで落ちます。

朝夕の山は通勤・通学の時間帯と重なるので、利用の中心は日常の移動と考えられます。

日次データの合計（1日平均4504台）だけでは見えない「時間帯の偏り」があるので、自転車の配置や整備はこの山に合わせて計画するのがよさそうです。


---


## 発展（任意）


### 自動EDAレポート（data-profiling）

今日は`describe()`・ヒストグラム・相関を1つずつ自分で出しました。列が12本ならこれで十分ですが、実務のデータは列が数十〜数百本あるのが普通で、1列ずつ眺めるのは現実的ではありません。

`data-profiling`（旧名ydata-profiling / pandas-profiling）は、データフレームを渡すだけで、各列の統計量・分布・欠損・相関・「注意すべき点（Alerts）」をまとめた1本のレポートを作ってくれるライブラリです。初めて見るデータの全体像を数分でつかむのに向いています。

ただし、あくまで「見当をつける」道具です。レポートは何が「変」かの候補を並べるだけで、それが本当に問題かどうかの結論は自分で出します。


In [ ]:
# Colab には入っていないので、なければインストールする（配布名は fg-data-profiling）
import importlib.util
if importlib.util.find_spec('data_profiling') is None and importlib.util.find_spec('ydata_profiling') is None:
    %pip install -q fg-data-profiling
# このライブラリは古い仕組み pkg_resources を使うので、無い環境（setuptools 81 以降）では補う
if importlib.util.find_spec('pkg_resources') is None:
    %pip install -q 'setuptools<81'


In [ ]:
# 新しい名前 data_profiling を優先し、旧名 ydata_profiling しかない環境でも動くようにする
try:
    from data_profiling import ProfileReport
except ImportError:
    from ydata_profiling import ProfileReport

# minimal=True で重い計算を省く。相関だけは見たいので、その計算だけ有効にする
prof = ProfileReport(df.drop(columns=['気温帯']), title='bike_day の自動EDA', minimal=True,
                     correlations={'auto': {'calculate': True}}, progress_bar=False)
prof.to_notebook_iframe()


読み方　レポートの上部にあるAlertsタブから見ます。`temp`と`atemp`に「High correlation」の警告が出ます。TODO①で自分の目で見つけた「そっくりな2列」を、道具も同じように拾ってきたわけです。

一方で`yr`・`holiday`・`weekday`・`workingday`には「Zeros」（ゼロが多い）の警告が出ます。これらは0/1や曜日番号の列なので、ゼロが多いのは当たり前で、問題ではありません。道具は機械的に候補を挙げるだけなので、こうした空振りを自分で見分けるのが人間の仕事です。

Variablesの項では列ごとの分布と統計量、Correlationsの項では相関の表が見られます（列の型に応じて指標が自動で選ばれるため、数値は5節と少し異なります）。Colabで重いと感じたら、`prof.to_file('report.html')`でファイルに書き出し、ブラウザで開く方法もあります。

試すなら、応用①で読んだ時間別データ`hour`を渡してみてください。行数が24倍あるので、少し時間がかかります。
